In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip install transformers datasets sentence-transformers torch scikit-learn -q

In [3]:
import warnings
warnings.filterwarnings('ignore')

import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import cos_sim
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [4]:
# q1
dataset = load_dataset('csv', data_files='/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv', split='train')

dataset = dataset.map(lambda x: {'combined_text': x['prompt'] + ' ' + x['A']})

length_index_51 = len(dataset[51]['combined_text'])
print("combined_text at index 51:")
print(dataset[51]['combined_text'])
print()
print("Character length:", length_index_51)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

combined_text at index 51:
Determine the correct option: What is the reason behind the designation of Class L dwarfs, and what is their color and composition? among the listed options. Class L dwarfs are hotter than M stars and are designated L because L is the remaining letter alphabetically closest to M. They are bright blue in color and are brightest in ultraviolet. Their atmosphere is hot enough to allow metal hydrides and alkali metals to be prominent in their spectra. Some of these objects have masses large enough to support hydrogen fusion and are therefore stars, but most are of substellar mass and are therefore brown dwarfs.

Character length: 614


In [5]:
# q2
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

vocab_size = tokenizer.vocab_size
print("BERT vocab size:", vocab_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

BERT vocab size: 30522


In [6]:
sep_token_id = tokenizer.sep_token_id
print("SEP token:", tokenizer.sep_token)
print("SEP token ID:", sep_token_id)

SEP token: [SEP]
SEP token ID: 102


In [7]:
# q4
# all_prompts = dataset['prompt']

# encoded = tokenizer(
#     all_prompts,
#     padding='max_length',
#     truncation=True,
#     max_length=128,
#     return_tensors='pt'
# )

# print("input_ids shape:", encoded['input_ids'].shape)

ValueError: text input must be of type `str` (single example), `list[str]` (batch or single pretokenized example) or `list[list[str]]` (batch of pretokenized examples) or `list[tuple[list[str], list[str]]]` (batch of pretokenized sequence pairs).

In [8]:
# q5
hidden_size = 768
num_heads = 12

head_dim = hidden_size // num_heads
print("Each attention head dimension:", head_dim)

Each attention head dimension: 64


In [9]:
# q6
model = AutoModel.from_pretrained('bert-base-uncased')
model.eval()

row0_prompt = dataset[0]['prompt']
print("Prompt:", row0_prompt)
print()

inputs = tokenizer(row0_prompt, return_tensors='pt')
print("Number of tokens (seq_len):", inputs['input_ids'].shape[1])

with torch.no_grad():
    outputs = model(**inputs)

print("last_hidden_state shape:", outputs.last_hidden_state.shape)
# Shape: (1, seq_len, 768)  ->  batch=1, tokens, hidden_dim

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Prompt: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.

Number of tokens (seq_len): 31
last_hidden_state shape: torch.Size([1, 31, 768])


In [10]:
# q7
cls_vector = outputs.last_hidden_state[0, 0, :]

first_5 = cls_vector[:5]
print("First 5 values:", first_5)
print("Sum of first 5 (rounded to 4dp):", round(first_5.sum().item(), 4))

First 5 values: tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480])
Sum of first 5 (rounded to 4dp): -1.2001


In [11]:
# q8
model_attn = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
model_attn.eval()

text = "Light-ion fusion is a technique."
inputs2 = tokenizer(text, return_tensors='pt')

tokens = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])
print("Tokens:", tokens)

fusion_idx = tokens.index('fusion')
print("Index of 'fusion':", fusion_idx)

with torch.no_grad():
    out2 = model_attn(**inputs2)

# out2.attentions is a tuple of 12 layers
# Each layer shape: (batch, num_heads, seq_len, seq_len)
last_layer = out2.attentions[-1]        # layer 12 (index -1)
head_0 = last_layer[0, 0]              # batch=0, head=0 -> (seq_len, seq_len)

cls_to_fusion = head_0[0, fusion_idx]  # row=CLS(0), col=fusion
print("Attention weight [CLS] -> 'fusion':", round(cls_to_fusion.item(), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
Index of 'fusion': 4
Attention weight [CLS] -> 'fusion': 0.1025


In [12]:
# q9
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

row0 = dataset[0]
prompt_text = row0['prompt']
option_b    = row0['B']

prompt_emb = st_model.encode(prompt_text, convert_to_tensor=True)
optb_emb   = st_model.encode(option_b,    convert_to_tensor=True)

similarity = cos_sim(prompt_emb, optb_emb)
print("Cosine similarity (prompt vs Option B, row 0):", round(similarity.item(), 4))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Cosine similarity (prompt vs Option B, row 0): 0.7658
